In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [11]:
import torch
import torch.nn as nn
import torch.functional as F
import numpy
import math

from tokenizer import CharacterTokenizer

### Tokenization and Embeddings

Tokenization will be done based on a character-level tokenizer.

In [12]:
text = "Hello! I am learning how to build a transformer from scratch using PyTorch."

tokenizer = CharacterTokenizer(text)

tokens = tokenizer.encode(text)

print(tokenizer.decode(tokens) == text)


True


The dimensions of the embeddings will be
$$
E_{sum}=E_{token}+E_{position}
$$

The shape of these tensors are as follows
- $E_{sum}$ : $x_1 + x_2 \in \mathbb{R^{T \times {d_{embed}}}}$
- $E_{token}$ and $E_{position}$ : $x \in \mathbb{R^{T \times {d_{embed}}}}$

where $T$ is the size of the sequence of tokens


In [13]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)
d_embed = 64

embedding_matrix = torch.randn(tokenizer.vocab_size,d_embed)
pos_embedding_matrix = torch.randn(128, d_embed)
positions = torch.arange(len(tokens))

embeddings = embedding_matrix[tokens]
embedded_tokens = pos_embedding_matrix[positions] + embeddings

print(pos_embedding_matrix.shape)
print(embedded_tokens.shape)
embedded_tokens

torch.Size([128, 64])
torch.Size([75, 64])


tensor([[-2.9129,  1.9816,  1.1873,  ..., -1.6824, -1.9508, -1.7453],
        [-1.4830, -0.8426,  2.8944,  ...,  0.9675,  0.2260,  0.1053],
        [ 3.3470,  0.5928,  0.0272,  ..., -0.6582,  0.1580, -1.1846],
        ...,
        [-0.1326, -2.0358, -0.7562,  ..., -0.5114,  0.7303, -0.1703],
        [-2.4227,  0.3152, -2.8114,  ...,  2.3804, -2.1081, -0.5127],
        [ 0.7059, -0.9471,  1.7307,  ...,  1.2132, -1.0064, -0.8405]])

### Self-Attention Mechanism

We compute the self-attention via computing 3 representations: $W_q, W_k$ and $W_v$



Then, we calculate the scaled dot product and apply a causal mask. The formula is:
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

After obtaining this result, we apply the softmax multiply by the value vector
$$
Attention(Q, K, V)=Softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

In [14]:
w_q, w_k, w_v = (torch.randn(64, 64), 
                 torch.randn(64, 64), 
                 torch.randn(64, 64))

query_vector = torch.matmul(embedded_tokens, w_q)
key_vector = torch.matmul(embedded_tokens, w_k)
value_vector = torch.matmul(embedded_tokens, w_v)

scaled_dot_product = torch.matmul(query_vector, key_vector.T) / math.sqrt(d_embed)

triu_rows, triu_cols = torch.triu_indices(row=scaled_dot_product.size(0), 
                                  col=scaled_dot_product.size(1),
                                  offset=1)

scaled_dot_product[triu_rows, triu_cols] = float("-inf")

scaled_dot_product.shape
soft_max = nn.Softmax(dim=1)

out = soft_max(scaled_dot_product)
print(out.sum(dim=1) == torch.ones(out.shape[0], out.shape[1]))

A = torch.matmul(out, value_vector)
print(f"Output Shape: {A.shape} \n{A}")



tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])
Output Shape: torch.Size([75, 64]) 
tensor([[ -5.6997,  -7.4701,  -1.1455,  ...,  10.7510,   6.0381,  10.2866],
        [ -6.0792, -16.7536,  18.9550,  ...,  19.7379,  23.1576,  -5.2264],
        [ -6.0792, -16.7536,  18.9550,  ...,  19.7379,  23.1576,  -5.2264],
        ...,
        [  8.8684,   8.3359,  -3.0672,  ..., -11.6554, -19.6224,   5.2486],
        [  1.4557,  -1.6267,  -4.7862,  ..., -12.7340, -14.7760,   6.7802],
        [-17.2633,   5.2881,   1.0734,  ...,   8.3863,   7.7567,  -3.6914]])


### Multi-Head Attention

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, count_heads, d_embed):
        super().__init__()
        self.heads = count_heads
        self.d_embed = d_embed
        self.d_head = d_embed // count_heads

        self.w_q = nn.ParameterList(
            [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])

        self.w_k = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])

        self.w_v = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])
        
        self.wO = nn.Parameter(torch.randn(d_embed, d_embed))


    def forward(self, X):
        query_vector: list = [torch.matmul(X, w_q) for w_q in self.w_q]
        key_vector: list = [torch.matmul(X, w_k) for w_k in self.w_k]
        value_vector: list = [torch.matmul(X, w_v) for w_v in self.w_v]
        T: int = X.size(0)

        head_outputs = [self.__attention(query_vector[i], key_vector[i], value_vector[i], T) for i in range(self.heads)]
        attention_concat = torch.cat(head_outputs, dim=1)

        return torch.matmul(attention_concat, self.wO)


    def __attention(self, q_i, k_i, v_i, T):
        scaled_dot_product = torch.matmul(q_i, k_i.T) / math.sqrt(self.d_head)

        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool),
            diagonal=1
        )
        masked_scores = scaled_dot_product.masked_fill(mask, float("-inf"))

        soft_max = nn.Softmax(dim=1)
        out = soft_max(masked_scores)

        attention_matrix = torch.matmul(out, v_i)

        return attention_matrix



In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiHeadAttention(4, 64).to(device)
print(model)

out = model(embedded_tokens)
print(f"Output Shape: {out.shape}\n{out}")

MultiHeadAttention(
  (w_q): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_k): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_v): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
)
Output Shape: torch.Size([75, 64])
tensor([[  21.1303,  -20.3785,  130.4431,  ...,   62.9390,  -23.6680,
          -11.2680],
        [  7

### Feed-Forward and Layer Normalization

In [17]:
class FeedForward(nn.Module):
    def __init__(self, d_embed):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_embed, 256),
            nn.ReLU(),
            nn.Linear(256, d_embed)
        )

    def forward(self, X):
        out = self.fc(X)
        return out


class LayerNorm(nn.Module):
    def __init__(self, d_embed, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_embed))
        self.beta = nn.Parameter(torch.zeros(d_embed))
        self.eps = eps

    def forward(self,x):
        mean = torch.mean(x, dim=1, keepdim=True)
        variance = torch.var(x, dim=1, keepdim=True, correction=0)

        return ((x - mean) / torch.sqrt(variance + self.eps)) * self.gamma + self.beta



### Transformer 

In [20]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, d_embed):
        super().__init__()
        self.attention = MultiHeadAttention(num_heads, d_embed)
        self.ffn = FeedForward(d_embed)
        
        self.norm1 = LayerNorm(d_embed)
        self.norm2 = LayerNorm(d_embed)

    def forward(self, X):
        x1 = X + self.attention(self.norm1(X))
        x2 = x1 + self.ffn(self.norm2(x1))

        return x2


In [22]:
transform = TransformerBlock(4, 64)
output = transform(embedded_tokens)

print(f"Embedded Tokens Shape: {embedded_tokens.shape}\nTransform Output Shape: {output.shape}")

Embedded Tokens Shape: torch.Size([75, 64])
Transform Output Shape: torch.Size([75, 64])
